# M9: Continuous Batching

M8 kept a batch together until every request in it ended. A request that finished early left its slot empty, and new requests waited for the whole batch. M9 decides the batch again on every step.

```text
get_next_batch_to_run   1. filter: drop requests that ended last step
        │               2. merge:  get_new_batch_prefill() fills the free slots, oldest waiting first
        │               3. return running_batch, or None when it is empty
        ▼
ModelRunner.forward     same left-padded [B, T] batch as M8
        ▼
process_batch_result    end_req() for the ones that finished; they stay in running_batch until step 1
```

The model runner, the sampler, streaming, and abort are unchanged. Only `get_next_batch_to_run()` and `get_new_batch_prefill()` are new, and one extra check in `abort_request()`.

## Setup

As in M8, `Engine` only loads the tokenizer and the model. Each experiment uses a fresh `Scheduler`. `run_continuous` records the rids of every step's batch and can call `on_step` inside each forward pass, so an experiment can submit a new request or an abort at an exact step.

In [1]:
import random
import statistics
import threading
import time

from engine import Engine
from req import FINISH_LENGTH, AbortReq, Req, ReqStatus
from scheduler import Scheduler

engine = Engine()
runner = engine.model_runner
tok = engine.tokenizer
print(runner.device)


def make_req(rid: str, prompt: str, max_new_tokens: int, eos: bool = True) -> Req:
    # eos=False: ignore EOS, so the request runs exactly max_new_tokens steps.
    return Req(
        rid=rid,
        prompt=prompt,
        input_ids=tok.encode(prompt),
        max_new_tokens=max_new_tokens,
        eos_token_ids=runner.eos_token_ids if eos else set(),
    )


def run_continuous(reqs: list[Req], max_batch_size: int, on_step=None) -> tuple[Scheduler, list[list[str]]]:
    # Queue every request first, then start the loop. Returns the scheduler and the rids in each step's batch.
    scheduler = Scheduler(runner, max_batch_size=max_batch_size)
    steps = []
    run_batch = scheduler.run_batch

    def logged(batch):
        steps.append([req.rid for req in batch])
        if on_step is not None:
            on_step(len(steps), batch, scheduler)
        return run_batch(batch)

    scheduler.run_batch = logged
    for req in reqs:
        scheduler.submit(req)
    threading.Thread(target=scheduler.event_loop, daemon=True).start()
    for req in reqs:
        req.done.wait()
    return scheduler, steps


def show_steps(steps: list[list[str]], max_batch_size: int) -> None:
    # One line per step; "+X" joined this step, "-X" left since the last one.
    prev = []
    for i, rids in enumerate(steps, 1):
        slots = " ".join(rids + ["·"] * (max_batch_size - len(rids)))
        changes = [f"+{r}" for r in rids if r not in prev] + [f"-{r}" for r in prev if r not in rids]
        print(f"step {i:>2}: {slots:<{2 * max_batch_size}}  {' '.join(changes)}")
        prev = rids
    used = sum(len(rids) for rids in steps)
    print(f"\nslots used: {used} / {len(steps) * max_batch_size} ({used / (len(steps) * max_batch_size):.0%})")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

mps


## Free Slots Refill on the Next Step

The same workload as the end of the M8 notebook: four requests with EOS ignored, so each runs exactly `max_new_tokens` steps, and `max_batch_size=3`.

In M8 this took 14 steps and used 19 of 42 slots (45%). D started at step 11, though a slot was free from step 3.

In [2]:
lengths = {"A": 2, "B": 10, "C": 3, "D": 4}
reqs = [make_req(rid, "Tell me a story.", n, eos=False) for rid, n in lengths.items()]
_, steps = run_continuous(reqs, max_batch_size=3)

show_steps(steps, max_batch_size=3)
print(f"D started at step {next(i for i, rids in enumerate(steps, 1) if 'D' in rids)}")

step  1: A B C   +A +B +C
step  2: A B C   
step  3: B C D   +D -A
step  4: B D ·   -C
step  5: B D ·   
step  6: B D ·   
step  7: B · ·   -D
step  8: B · ·   
step  9: B · ·   
step 10: B · ·   

slots used: 19 / 30 (63%)
D started at step 3


A produces its last token in step 2 and ends there, but it is still in `running_batch` when step 2 finishes. At the start of step 3, the filter drops A and D takes the free slot. The work is the same 19 request-steps, now done in 10 steps instead of 14. After C and D end, B runs alone: with no request waiting, there is nothing to fill the slots with.

A slot is capacity, not a position. Filtering keeps the order of the requests that remain, and new requests go at the end, so D appears after B and C, not where A was.

## Inside One Step: Filter, Then Merge

`get_next_batch_to_run()` needs no model and no thread, so it can be driven by hand. Build the state that the previous step leaves behind: `done` ended with `end_req()` but is still in `running_batch`, `running` is still generating, and three requests wait.

In [3]:
scheduler = Scheduler(runner, max_batch_size=3)
done, running = make_req("done", "Hi", 1), make_req("running", "Hi", 5)
for req in (done, running):
    req.set_status(ReqStatus.RUNNING)
done.finished_reason = FINISH_LENGTH(1)
done.set_status(ReqStatus.FINISHED)
scheduler.running_batch = [done, running]
scheduler.waiting_queue.extend(make_req(f"W{i}", "Hi", 1) for i in (1, 2, 3))


def state(label: str) -> None:
    running_batch = [f"{r.rid}({r.status.name})" for r in scheduler.running_batch]
    waiting = [r.rid for r in scheduler.waiting_queue]
    print(f"{label:<7} running_batch={running_batch}  waiting_queue={waiting}")


state("before")
batch = scheduler.get_next_batch_to_run()
state("after")
print("batch is running_batch:", batch is scheduler.running_batch)

# Everything ended and nothing waits: no batch.
idle = Scheduler(runner)
idle.running_batch = [done]
print("idle step:", idle.get_next_batch_to_run(), idle.running_batch)

before  running_batch=['done(FINISHED)', 'running(RUNNING)']  waiting_queue=['W1', 'W2', 'W3']
after   running_batch=['running(RUNNING)', 'W1(RUNNING)', 'W2(RUNNING)']  waiting_queue=['W3']
batch is running_batch: True
idle step: None []


`done` is dropped, one slot opens next to `running`, and the oldest two waiting requests take the two free slots. W3 stays in `waiting_queue` and is marked RUNNING only when it joins. When nothing is left, `get_next_batch_to_run()` returns `None`, and the event loop blocks on `recv_queue` instead of running an empty forward pass.

## A Newcomer Runs Its Whole Prompt

There is no KV cache yet, so every step feeds each request's whole sequence, prompt plus every token generated so far. A request that joins mid-run is padded into the same `[B, T]` tensor as the requests already generating.

Below, A has a long prompt and B a short one, with `max_batch_size=2`. C joins when B leaves. For each step, print the tensor shape and the real tokens per row.

In [4]:
shapes = []


def record_shape(step, batch, scheduler):
    _, attention_mask, _ = runner.prepare_inputs(batch)
    real = [f"{req.rid}={int(row.sum())}" for req, row in zip(batch, attention_mask, strict=True)]
    shapes.append(f"step {step}: {tuple(attention_mask.shape)}  real tokens {' '.join(real)}")


reqs = [
    make_req("A", "In the beginning of the story, a young dragon lived alone on a quiet mountain", 6, eos=False),
    make_req("B", "Hi", 2, eos=False),
    make_req("C", "The capital of France is", 3, eos=False),
]
run_continuous(reqs, max_batch_size=2, on_step=record_shape)
print("\n".join(shapes))

step 1: (2, 16)  real tokens A=16 B=1
step 2: (2, 17)  real tokens A=17 B=2
step 3: (2, 18)  real tokens A=18 C=5
step 4: (2, 19)  real tokens A=19 C=6
step 5: (2, 20)  real tokens A=20 C=7
step 6: (1, 21)  real tokens A=21


Every row grows by one token per step, and every step recomputes all of them. In step 3, C's whole prompt and A's prompt plus two generated tokens run in the same forward pass. Without a KV cache this mix costs nothing extra, because every row is recomputed in full anyway.

## Joining Mid-Run Keeps the Same Tokens

The contract of M8 still holds: batching must not change any request's output. Now requests join and leave at different steps. A, B, C are queued at the start with `max_batch_size=2`, so C waits for a slot. D is submitted during step 2, while the scheduler thread is inside a forward pass, and joins on a later step. Compare each request with running it alone.

In [5]:
prompts = {
    "A": ("The capital of France is", 3),
    "B": ("Write a haiku about the sea.", 10),
    "C": ("1, 2, 3, 4,", 6),
    "D": ("Hi", 4),
}

alone = {}
for rid, (prompt, n) in prompts.items():
    req = make_req(rid, prompt, n, eos=False)
    run_continuous([req], max_batch_size=1)
    alone[rid] = req.output_ids

together = {rid: make_req(rid, prompt, n, eos=False) for rid, (prompt, n) in prompts.items()}


def submit_d(step, batch, scheduler):
    if step == 2:
        scheduler.submit(together["D"])


_, steps = run_continuous([together[rid] for rid in "ABC"], max_batch_size=2, on_step=submit_d)
together["D"].done.wait()

show_steps(steps, max_batch_size=2)
print()
for req in together.values():
    print(f"{req.rid}: same={req.output_ids == alone[req.rid]}  {tok.decode(req.output_ids)!r}")

step  1: A B   +A +B
step  2: A B   
step  3: A B   
step  4: B C   +C -A
step  5: B C   
step  6: B C   
step  7: B C   
step  8: B C   
step  9: B C   
step 10: B D   +D -C
step 11: D ·   -B
step 12: D ·   
step 13: D ·   

slots used: 23 / 26 (88%)

A: same=True  ' Paris. The'
B: same=True  ' The sea is a place where people come to live'
C: same=True  ' 5, 6,'
D: same=True  'Question\nOkay,'


D arrives while A and B are generating, so it goes to `waiting_queue` behind C. When A ends, C takes the slot; D waits until the next one opens. FIFO order holds even for a request that arrives mid-run.

As in M8, `same=False` on MPS or CUDA can come from bf16 rounding: a different batch shape can flip a near-tie between two tokens. Here the batch shape changes whenever a request joins or leaves. The tests check this on CPU.

## An Abort That Arrives Too Late

An ended request stays in `running_batch` until the next filter. An abort that arrives in between finds it there, and aborting it again would be an illegal `FINISHED -> ABORTED` transition. So `abort_request()` skips requests that already ended.

A has 2 tokens to produce and B has 5. During step 2, A's last forward pass, submit an abort for both. The scheduler reads both aborts at the start of step 3. A ended in step 2, so its abort is ignored. B is still generating, so it is aborted.

In [6]:
a = make_req("A", "Hello", 2, eos=False)
b = make_req("B", "Hello", 5, eos=False)


def abort_both(step, batch, scheduler):
    if step == 2:
        scheduler.submit(AbortReq("A"))
        scheduler.submit(AbortReq("B"))


_, steps = run_continuous([a, b], max_batch_size=2, on_step=abort_both)
print("steps:", steps)
for req in (a, b):
    print(f"{req.rid}: {req.status.name:<8} {req.finished_reason.to_json()}  tokens={len(req.output_ids)}")

steps: [['A', 'B'], ['A', 'B']]
A: FINISHED {'type': 'length', 'length': 2}  tokens=2
B: ABORTED  {'type': 'abort', 'message': 'Aborted'}  tokens=2


## Throughput: Static vs Continuous

To compare the two policies on the same code, rebuild M8's policy as a subclass. The only difference is one condition: admit new requests only when every request in the batch has ended.

For each run, measure:

- **tokens/s**: generated tokens divided by wall time, from the first submit to the last request ending.
- **TTFT** (time to first token): from submit to the end of the request's first step. Queueing time counts.
- **latency**: from submit to the request's end.
- **fed tokens**: the sum of `B × T` over all steps, including padding, next to the real tokens without padding.

Two workloads, both with EOS ignored so the output lengths are fixed:

- **skewed**: 32 requests, prompt `"Hi"`, every 8th request generates 32 tokens and the rest generate 4. `max_batch_size=8`.
- **uniform**: 16 requests, prompt `"Tell me a story."`, lengths drawn from 2 to 32. `max_batch_size=4`.

In [7]:
class StaticScheduler(Scheduler):
    # M8's policy: admit new requests only when the whole batch has ended.
    def get_next_batch_to_run(self):
        self.running_batch = [req for req in self.running_batch if not req.finished()]
        if not self.running_batch:
            self.running_batch = self.get_new_batch_prefill()
        return self.running_batch or None


def measure(scheduler_cls, reqs: list[Req], max_batch_size: int) -> None:
    scheduler = scheduler_cls(runner, max_batch_size=max_batch_size)
    first_token, ended = {}, {}
    steps, fed, real = 0, 0, 0
    run_batch, end_req = scheduler.run_batch, scheduler.end_req

    def timed_run_batch(batch):
        nonlocal steps, fed, real
        lengths = [len(req.input_ids) + len(req.output_ids) for req in batch]
        steps, fed, real = steps + 1, fed + len(batch) * max(lengths), real + sum(lengths)
        result = run_batch(batch)
        now = time.perf_counter()
        for req in batch:
            first_token.setdefault(req.rid, now)
        return result

    def timed_end_req(req, status):
        ended[req.rid] = time.perf_counter()
        end_req(req, status)

    scheduler.run_batch, scheduler.end_req = timed_run_batch, timed_end_req
    start = time.perf_counter()
    for req in reqs:
        scheduler.submit(req)
    threading.Thread(target=scheduler.event_loop, daemon=True).start()
    for req in reqs:
        req.done.wait()
    total = time.perf_counter() - start

    tokens = sum(len(req.output_ids) for req in reqs)
    ttft = statistics.mean(first_token[req.rid] - start for req in reqs)
    latency = statistics.mean(ended[req.rid] - start for req in reqs)
    print(
        f"  {scheduler_cls.__name__:<15} steps {steps:>3}  fed {fed:>5} (real {real:>5})  "
        f"time {total:5.2f} s  tokens/s {tokens / total:6.1f}  TTFT {ttft:5.2f} s  latency {latency:5.2f} s"
    )


def skewed() -> list[Req]:
    ids = tok.encode("Hi")
    return [Req(rid=str(i), prompt="", input_ids=ids, max_new_tokens=32 if i % 8 == 0 else 4) for i in range(32)]


def uniform() -> list[Req]:
    rng = random.Random(0)
    ids = tok.encode("Tell me a story.")
    return [Req(rid=str(i), prompt="", input_ids=ids, max_new_tokens=rng.randint(2, 32)) for i in range(16)]


measure(Scheduler, uniform()[:4], max_batch_size=4)  # warm up the device; not printed below
print()
for name, make, max_batch_size in [("skewed", skewed, 8), ("uniform", uniform, 4)]:
    print(f"{name}, max_batch_size={max_batch_size}")
    for scheduler_cls in (StaticScheduler, Scheduler):
        measure(scheduler_cls, make(), max_batch_size)

  Scheduler       steps  30  fed  1752 (real  1752)  time  2.89 s  tokens/s   34.2  TTFT  0.06 s  latency  2.35 s

skewed, max_batch_size=8
  StaticScheduler steps 128  fed  2392 (real  2392)  time  6.69 s  tokens/s   35.9  TTFT  2.64 s  latency  3.06 s
  Scheduler       steps  44  fed  3864 (real  2392)  time  5.78 s  tokens/s   41.5  TTFT  0.84 s  latency  1.78 s
uniform, max_batch_size=4
  StaticScheduler steps 125  fed  5716 (real  5716)  time 10.26 s  tokens/s   32.8  TTFT  3.84 s  latency  5.59 s
  Scheduler       steps 104  fed  7407 (real  5716)  time 11.52 s  tokens/s   29.3  TTFT  3.55 s  latency  5.98 s


The first line is the warm-up run. Each shape the device sees for the first time is slower, so its numbers are not part of the comparison.

**skewed**: every static batch waits for its one long request, and runs with a single row for most of its steps. Continuous batching refills those slots at once, so it needs about a third of the steps. Throughput goes up, and TTFT and latency drop a lot, because short requests no longer wait behind long ones.

**uniform**: continuous batching still needs fewer steps and gives a slightly lower TTFT, but throughput and mean latency can be the same or worse. Two reasons, both caused by the missing KV cache:

- The real tokens are the same for both policies. Each step recomputes each request's whole sequence, so the cost of a request depends only on its own length, not on when it runs. Scheduling cannot remove any of that work.
- Continuous batching feeds more padding. A static batch starts together, so here all its rows have the same length. A continuous batch mixes a long-running request with newcomers, and every row is padded to the longest one.

So fewer steps only help when a step's time barely depends on how many rows it holds. With short sequences (skewed) the device is mostly waiting on memory, and extra rows are almost free. With longer rows and no KV cache, every row adds real compute. A KV cache makes each generating row one token per step, which is the case where continuous batching pays off most.

Timings depend on the device and change from run to run; compare the policies within one run.

**Something to think about.** Here a newcomer's prompt runs in the same forward pass as requests that are generating. Once a KV cache exists, a generating request feeds one new token per step, but a newcomer still feeds its whole prompt. What happens to the step time of every generating request when a 2000-token prompt joins? SGLang runs new requests in their own prefill batch first, and can split a long prompt into chunks (`--chunked-prefill-size`).

## Summary

| | M8 | M9 |
|---|---|---|
| When the batch changes | only when `running_batch` is empty | every step |
| Ended requests | `end_req()` removes them at once | stay in `running_batch` until the next `get_next_batch_to_run()` drops them |
| Free slots | stay empty until the whole batch ends | filled on the next step, oldest waiting first |
| Request that arrives mid-run | waits for the whole batch | joins when a slot opens |
| Abort of a request that just ended | not in `running_batch`, ignored | still in `running_batch`; skipped because `finished()` |
| Model input | left-padded `[B, T]` | same; a newcomer's full prompt shares the pass with generating rows |
| Output | token-identical to running alone | same, even when requests join and leave |
| Throughput | a batch runs as long as its longest request | fewer steps and lower TTFT; more tokens/s when extra rows are cheap |

Continuous batching changes only the scheduler. The model runner and the sampler do not know that the batch is rebuilt every step.

**Compare with SGLang**: `filter_batch()` and `merge_batch()` in `managers/schedule_batch.py`, `get_next_batch_to_run()` and `update_running_batch()` in `managers/scheduler.py`.